### Kafka to Demo Real-Time Fraud and Anomaly Detection

The code below utilizes an internal memory-rate driver to generate simulated transactions, formats them into Kafka-compliant records (a schema containing a key and a value), writes them to an in-memory cluster channel, and reads them back via your sliding-window aggregator pipeline.

### Learning Objectives

Parallel Partitions & Tasks: Students will see how Spark maps Kafka partitions to independent, distributed tasks.

Real-time Stream Aggregations: How to perform stateful sliding-window computations on high-throughput data.

Immutability and Transformations: Moving raw bytes into highly structured data streams.

### Producer

Instead of requiring an external Kafka cluster setup, this script utilizes a self-contained data generator thread inside the same process. It injects a mix of normal transactions and deliberate "anomalous spikes" directly into a mock Kafka broker or topic, which your students can immediately visualize processing across parallel stream channels.

What to Point Out to Students During the Demo

**The master("local[*]") Parameter:** Explain that this isn't just a sequential Python script. Spark is dividing the execution engine into multiple JVM worker threads on their laptop, imitating a real cloud cluster environment.

**The "Anomaly Detected" Flag:** Every few seconds, the fallback engine artificially mimics a credit card fraud spike (amount > $5000). Students can watch the live terminal update dynamically as distributed tasks detect and flag this state across independent streaming windows.

**Micro-batch Triggers:** Watch how the data refreshes at predictable time intervals. You can discuss the architectural trade-offs between Spark's Micro-batch processing paradigm vs. continuous processing frameworks like Apache Flink.

### Code

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, window, count, avg, max, expr
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

In [12]:
# 1. INITIALIZE PYSPARK 4.0.0 CLUSTER
spark = SparkSession.builder \
    .appName("Distributed-Computing-Class-Demo") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

print(f"=== Spark Worker Threads Spinning Up... Version: {spark.version} ===")

=== Spark Worker Threads Spinning Up... Version: 4.0.0 ===


In [13]:
# -------------------------------------------------------------------------
# 2. THE DISTRIBUTED PRODUCER (Simulated Event Generation)
# -------------------------------------------------------------------------
# Generates 10 transactions per second natively using Spark workers
simulated_network_traffic = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", "10") \
    .load()

# Transform raw row IDs into a JSON format matching real edge devices
# Deliberately injects an anomaly (amount = $5,000) every 15 records
kafka_formatted_producer = simulated_network_traffic.selectExpr(
    "CAST(value AS STRING) as key",
    """CAST(
        to_json(named_struct(
            'terminal_id', concat('TERM-', cast((value % 5) as string)),
            'region', case when value % 3 == 0 then 'US-EAST' when value % 3 == 1 then 'EU-WEST' else 'APAC' end,
            'amount', cast((value % 100) + case when value % 15 == 0 then 5000.0 else 15.5 end as double),
            'tx_timestamp', timestamp
        )) AS STRING
    ) as value"""
)

# -------------------------------------------------------------------------
# ARCHITECTURE BREAKPOINT FOR STUDENTS:
# At this line, the 'kafka_formatted_producer' dataframe holds an explicit 
# Kafka schema mapping: a string 'key' and a string JSON 'value' payload.
# -------------------------------------------------------------------------


# -------------------------------------------------------------------------
# 3. THE CONSUMER (Streaming Pipeline & Deserialization)
# -------------------------------------------------------------------------
transaction_schema = StructType([
    StructField("terminal_id", StringType(), True),
    StructField("region", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("tx_timestamp", TimestampType(), True)
])

# Read the stream straight out of our producer dataframe
# We extract the 'value' payload and break the JSON apart into distinct typed columns
structured_consumer = kafka_formatted_producer \
    .selectExpr("CAST(value AS STRING) as json_payload") \
    .withColumn("data", from_json(col("json_payload"), transaction_schema)) \
    .select("data.*")


# -------------------------------------------------------------------------
# 4. COMPUTER SCIENCE CONCEPT: STATEFUL SLIDING WINDOW AGGREGATION
# -------------------------------------------------------------------------
# Computes aggregations over a 10-second window that recalculates every 5 seconds
aggregated_dashboard = structured_consumer \
    .groupBy(
        window(col("tx_timestamp"), "10 seconds", "5 seconds"),
        col("region")
    ) \
    .agg(
        count("amount").alias("tx_count"),
        avg("amount").alias("average_spend"),
        max("amount").alias("peak_transaction")
    ) \
    .withColumn("FRAUD_ALERT", expr("peak_transaction > 4000.0")) \
    .sort(col("window.start").desc())


# -------------------------------------------------------------------------
# 5. STREAM SINK (Launch Processing Topologies)
# -------------------------------------------------------------------------
print("--> Engaging Stream Engine. Watch micro-batches execute live below:")

query = aggregated_dashboard.writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", "false") \
    .start()

try:
    query.awaitTermination()
except KeyboardInterrupt:
    print("\nStopping streaming cluster topology...")
finally:
    spark.stop()


--> Engaging Stream Engine. Watch micro-batches execute live below:


26/08/31 17:35:52 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-8b50dd3f-3503-40d8-9ca8-f093d61c0e57. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/08/31 17:35:52 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+------+------+--------+-------------+----------------+-----------+
|window|region|tx_count|average_spend|peak_transaction|FRAUD_ALERT|
+------+------+--------+-------------+----------------+-----------+
+------+------+--------+-------------+----------------+-----------+

-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+-------+--------+------------------+----------------+-----------+
|window                                    |region |tx_count|average_spend     |peak_transaction|FRAUD_ALERT|
+------------------------------------------+-------+--------+------------------+----------------+-----------+
|{2026-08-31 17:35:55, 2026-08-31 17:36:05}|APAC   |9       |50.5              |62.5            |false      |
|{2026-08-31 17:35:55, 2026-08-31 17:36:05}|EU-WEST|9       |52.5              |64.5         

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/apps/software/standard/core/pyspark/4.0.0-py3.11/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/apps/software/standard/core/pyspark/4.0.0-py3.11/lib/python3.11/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/apps/software/standard/core/pyspark/4.0.0-py3.11/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
26/08/31 17:37:20 ERROR MicroBatchExecution: Query [id = e9502343-8f29-48d8-a69e-58bb04382250, runId = 40d12536-dcb3-49ad-b52c-9f975601047c] terminated with error
org.apache.spark.SparkException: [INTERNAL_ERROR] The Spark SQL phase optimization


Stopping streaming cluster topology...


26/08/31 17:37:56 WARN StateStore: Error running maintenance thread
java.lang.IllegalStateException: SparkEnv not active, cannot do maintenance on StateStores
	at org.apache.spark.sql.execution.streaming.state.StateStore$.doMaintenance(StateStore.scala:950)
	at org.apache.spark.sql.execution.streaming.state.StateStore$.$anonfun$startMaintenanceIfNeeded$1(StateStore.scala:924)
	at org.apache.spark.sql.execution.streaming.state.StateStore$MaintenanceTask$$anon$1.run(StateStore.scala:725)
	at java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:572)
	at java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:358)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1144)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:642)
	at java.base/java.lang.Thread.